<a href="https://colab.research.google.com/github/pyrenaaaaaa/Emotion-Aware-Companion/blob/text/Llama_3_2_3B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# # Mount Google Drive
from google.colab import drive
drive. mount ('/content/drive')

Mounted at /content/drive


In [2]:
# Install Unsloth and dependencies (COLAB ONLY)
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Install only in Colab to avoid conflicts
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
    !pip install --no-deps cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

# Restart kernel after installing
import IPython
IPython.display.clear_output()

# Imports


In [3]:
# Ensure Unsloth is imported FIRST before Transformers, TRL, PEFT
import unsloth
from unsloth import FastLanguageModel

import torch
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from datasets import load_dataset, DatasetDict
from transformers import BitsAndBytesConfig, TextStreamer
from accelerate import infer_auto_device_map
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Check GPU info
gpu_info = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_info.name}, Total Memory: {gpu_info.total_memory / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4, Total Memory: 15.83 GB


# Configure LLaMA 3.3 Model with LoRA + 4-bit Quantization


In [6]:
from unsloth import FastLanguageModel
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM

# Configure 4-bit Quantization with CPU offloading
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # Memory-efficient
    bnb_4bit_compute_dtype=torch.float16,  # Use float16 for best performance on T4
    bnb_4bit_use_double_quant=True,  # Reduces memory further
)

# Define parameters
max_seq_length = 512  # Lower memory usage
dtype = torch.float16  # Use float16 instead of bfloat16
load_in_4bit = True  # Enable 4-bit quantization

# Load smaller LLaMA model (Avoid 70B, Use 3B or 1B)
model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"  # Choose based on GPU memory

# Load Model with Offloading
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    quantization_config=quantization_config,
    device_map="auto"  # Auto-offload layers between GPU & CPU
)

==((====))==  Unsloth 2025.3.9: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


# Prepare Dataset for Emotion Aware Training

In [7]:
# Load a conversation-style dataset (Replace with your dataset)
dataset = load_dataset("mlabonne/FineTome-100k", split="train")

# Convert dataset to HuggingFace's standard format
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(dataset)

# Define chat formatting function
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

# Check dataset structure
print(dataset[5]["text"])

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Standardizing format:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

How do astronomers determine the original wavelength of light emitted by a celestial body at rest, which is necessary for measuring its speed using the Doppler effect?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Astronomers make use of the unique spectral fingerprints of elements found in stars. These elements emit and absorb light at specific, known wavelengths, forming an absorption spectrum. By analyzing the light received from distant stars and comparing it to the laboratory-measured spectra of these elements, astronomers can identify the shifts in these wavelengths due to the Doppler effect. The observed shift tells them the extent to which the light has been redshifted or blueshifted, thereby allowing them to calculate the speed of the star along the line of sight relative to Earth.<|eot_id|>


# Train the Model

# Fine-Tune LLaMA 3.3 on Emotional Conversations

In [10]:
# Set up Data Collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    padding=True,
    max_length=max_seq_length,
    return_tensors="pt",
    label_pad_token_id=-100,
)

# ✅ Apply LoRA to enable fine-tuning on quantized model
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank (Higher = More trainable parameters, 8-64 suggested)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,  # Scaling factor
    lora_dropout=0,  # Keeps dropout optimized
    bias="none",  # Avoids unnecessary bias tuning
    use_gradient_checkpointing="unsloth",  # Enables memory-efficient training
    random_state=3407,
    use_rslora=False,  # Optional: Enables rank-stabilized LoRA
    loftq_config=None,  # Optional: Uses LoftQ for further compression
)

# Configure Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=data_collator,
    dataset_num_proc=2,  # Parallel processing
    packing=False,  # Prevents excessive memory usage
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=1000,  # Increase for better results
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

# Apply LoRA Training Optimizations
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

# Start Training
trainer.train()

Unsloth: We found double BOS tokens - we shall remove one automatically.


Map (num_proc=8):   0%|          | 0/100000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,000 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856/1,827,777,536 (1.33% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.894100
20,0.850500
30,0.859500
40,0.750700
50,0.876800
60,0.879300
70,0.860300
80,0.815800
90,0.831400
100,0.734800


Step,Training Loss
10,0.894100
20,0.850500
30,0.859500
40,0.750700
50,0.876800
60,0.879300
70,0.860300
80,0.815800
90,0.831400
100,0.734800


TrainOutput(global_step=1000, training_loss=0.7620418863296509, metrics={'train_runtime': 5648.5922, 'train_samples_per_second': 1.416, 'train_steps_per_second': 0.177, 'total_flos': 6.444071252471808e+16, 'train_loss': 0.7620418863296509})

# Run Inference on Emotion Aware Chatbot

In [13]:
# Run Inference
FastLanguageModel.for_inference(model)  # Enable optimized inference

messages = [
    {"role": "user", "content": "I'm feeling really stressed today. What should I do?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate response
outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True, temperature=1.5)

# Remove system prompts & format properly
generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
generated_text = generated_text.split("assistant\n\n")[-1]  # Extract assistant's response only

print("**Model Response:**", generated_text)

**Model Response:** Hey there, take a deep breath and do some simple yoga poses to reduce your stress levels. Also, try eating a light snack.


# Save & Export Model

In [ ]:
save_path = "/content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}!")